# Selective replication on GCP
Run one cell at a time: **settings → create → setup → smoke → full → download → cleanup**.
Every new session creates a fresh cluster from the saved image. No old state file is needed.
The GCP commands are below so you can edit them directly. Training, scheduling and correctness checks remain in `selective_replication/`.
These tests stop worker VMs; the head and its checkpoint storage survive.

In [ ]:
import concurrent.futures
import json
import shlex
import subprocess
import sys
import time
import uuid
from pathlib import Path

PROJECT = "research-488322"
ZONE = "us-central1-a"
IMAGE = "ray-built-v1"
NETWORK = "default"  # Existing VPC; no new network is created.
SUBNET = "default"  # Existing subnet in the region of ZONE.
MACHINE_TYPE = "e2-standard-4"
DISK_GB = 100  # Must be at least as large as the saved image.
PYTHON = "/opt/ray-env/ray-dev/bin/python"

SMOKE_EPOCHS = 1200
SMOKE_CHECKPOINT_EVERY = 100
FULL_EPOCHS = 2400
FULL_CHECKPOINT_EVERY = 200
REPEATS = 5
CASES = None  # None = all eight price/risk/deadline settings.
PRICE_MULTIPLIERS = [0.25, 1.0]
FAILURE_PROBABILITIES = [0.1, 0.5]
DEADLINE_MULTIPLIERS = [1.25, 2.0]
FAULT_TIME_FRACTIONS = [0.2, 0.7]
MAX_OBSERVATIONS = 1800
INSTALL_MISSING_DEPENDENCIES = True

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "gossip_benchmarks").is_dir())
HELPERS = ROOT / "gossip_benchmarks/selective_replication"
WORKDIR = ROOT / "ray_gcp_results"
sys.path.insert(0, str(HELPERS))

# Keep the same names if this cell is rerun in the current kernel.
# Restart the kernel to start a new experiment with new names.
if "RUN_ID" not in globals():
    RUN_ID = "rayexp-" + time.strftime("%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
NAMES = [RUN_ID + "-head"] + [RUN_ID + f"-w{i}" for i in range(1, 5)]
HEAD, *WORKERS = NAMES
REGION = ZONE.rsplit("-", 1)[0]
LOCAL = WORKDIR / RUN_ID
LOCAL.mkdir(parents=True, exist_ok=True)
STATE_FILE = LOCAL / "state.json"
print("VMs:", *NAMES, sep="\n  ")
print("Results:", LOCAL)


## 1. Small command helpers

In [ ]:
def gcloud(*args, timeout=600):
    command = ["gcloud", *map(str, args), f"--project={PROJECT}", "--quiet"]
    print(shlex.join(command), flush=True)
    result = subprocess.run(command, text=True, capture_output=True, timeout=timeout)
    if result.returncode:
        raise RuntimeError(result.stderr + "\n" + result.stdout)
    return result.stdout


def ssh(name, command):
    return gcloud("compute", "ssh", name, f"--zone={ZONE}", "--tunnel-through-iap",
                  "--ssh-flag=-oConnectTimeout=15", f"--command={command}")


def scp(source, target):
    return gcloud("compute", "scp", source, target, f"--zone={ZONE}", "--tunnel-through-iap")


def run_parallel(function, items):
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as pool:
        return list(pool.map(function, items))


## 2. Create fresh VMs
The CLI check runs before creating billable resources. Rerunning this cell after creation does not create another cluster.
Fresh VMs use the existing `NETWORK` and `SUBNET` selected above. No VPC is created. If `default` is absent, list networks/subnets with `gcloud compute networks list` and `gcloud compute networks subnets list`, then edit the two settings.

In [ ]:
from gcp.capabilities import stop_command

print("Failure command:", "gcloud", *stop_command(), "--no-graceful-shutdown")
if STATE_FILE.exists():
    print("Creation already recorded for this session. Continue with setup.", STATE_FILE)
else:
    image = json.loads(gcloud("compute", "images", "describe", IMAGE, "--format=json"))
    assert image["status"] == "READY"
    gcloud("compute", "networks", "describe", NETWORK, "--format=json")
    subnet = json.loads(gcloud("compute", "networks", "subnets", "describe", SUBNET,
                               f"--region={REGION}", "--format=json"))
    assert subnet["network"].rsplit("/", 1)[-1] == NETWORK, "Subnet belongs to a different VPC"
    # Record names before creation so partial failures can be cleaned up.
    STATE_FILE.write_text(json.dumps(dict(project=PROJECT, zone=ZONE, run_id=RUN_ID,
                                         names=NAMES, image=IMAGE, network=NETWORK, subnet=SUBNET), indent=2))
    gcloud("compute", "firewall-rules", "create", RUN_ID + "-internal",
           f"--network={NETWORK}", "--allow=tcp,udp,icmp",
           f"--source-tags={RUN_ID}", f"--target-tags={RUN_ID}")
    gcloud("compute", "firewall-rules", "create", RUN_ID + "-iap",
           f"--network={NETWORK}", "--allow=tcp:22",
           "--source-ranges=35.235.240.0/20", f"--target-tags={RUN_ID}")
    gcloud("compute", "instances", "create", *NAMES, f"--zone={ZONE}",
           f"--machine-type={MACHINE_TYPE}", f"--image={IMAGE}", f"--image-project={PROJECT}",
           f"--boot-disk-size={max(DISK_GB, int(image['diskSizeGb']))}GB",
           "--boot-disk-type=pd-balanced", "--provisioning-model=STANDARD",
           f"--subnet={SUBNET}", f"--tags={RUN_ID}", f"--labels=ray-experiment={RUN_ID}",
           "--metadata=enable-oslogin=FALSE", "--no-service-account", "--no-scopes", timeout=900)


## 3. Get IPs and wait for SSH

In [ ]:
nodes = []
for name in NAMES:
    vm = json.loads(gcloud("compute", "instances", "describe", name, f"--zone={ZONE}", "--format=json"))
    assert vm.get("labels", {}).get("ray-experiment") == RUN_ID, name
    assert vm["status"] == "RUNNING", (name, vm["status"])
    nodes.append(dict(name=name, ip=vm["networkInterfaces"][0]["networkIP"], head=name == HEAD))
HEAD_IP = nodes[0]["ip"]
print(json.dumps(nodes, indent=2))


def wait_ssh(name):
    for attempt in range(30):
        try:
            ssh(name, "true")
            return
        except (RuntimeError, subprocess.TimeoutExpired):
            if attempt == 29:
                raise
            time.sleep(5)


wait_ssh(HEAD)  # Initialize gcloud's local SSH key before parallel connections.
run_parallel(wait_ssh, WORKERS)


## 4. Install the node script and create the shared directory
The saved image already contains Ray and PyTorch. This cell installs only NFS tools and the small script that starts/stops Ray.

In [ ]:
def setup_node(node):
    name = node["name"]
    ssh(name, """set -e
sudo apt-get update -qq
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nfs-common
id raybench >/dev/null 2>&1 || sudo useradd --uid 25001 --create-home --home-dir /var/lib/raybench --shell /bin/bash raybench
test "$(id -u raybench)" = 25001
sudo mkdir -p /opt/ray-gcp /srv/ray-gcp
mountpoint -q /srv/ray-gcp || sudo chown raybench:raybench /srv/ray-gcp
""")
    path = LOCAL / (name + ".json")
    path.write_text(json.dumps(dict(node, nodes=nodes, head_ip=HEAD_IP, run_id=RUN_ID)))
    scp(path, name + ":/tmp/ray-node.json")
    scp(HELPERS / "gcp/bootstrap/node_control.py", name + ":/tmp/ray-node-control.py")
    ssh(name, """set -e
sudo install -o root -g root -m 644 /tmp/ray-node.json /etc/ray-gcp.json
sudo install -o root -g root -m 644 /tmp/ray-node-control.py /opt/ray-gcp/node_control.py
""")


run_parallel(setup_node, nodes)


## 5. Share checkpoints from the head
Mounts persist when workers stop and restart between observations.

In [ ]:
exports = "/srv/ray-gcp " + " ".join(
    n["ip"] + "(rw,sync,no_subtree_check,root_squash,fsid=0)" for n in nodes[1:]
)
ssh(HEAD, f"""set -e
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nfs-kernel-server
sudo mkdir -p /etc/exports.d
printf '%s\\n' {shlex.quote(exports)} | sudo tee /etc/exports.d/ray-gcp.exports >/dev/null
sudo exportfs -ra
sudo systemctl enable --now nfs-server
""")


def mount_worker(name):
    entry = f"{HEAD_IP}:/ /srv/ray-gcp nfs4 vers=4.2,hard,_netdev 0 0"
    ssh(name, f"""set -e
grep -qF {shlex.quote(entry)} /etc/fstab || printf '%s\\n' {shlex.quote(entry)} | sudo tee -a /etc/fstab >/dev/null
mountpoint -q /srv/ray-gcp || sudo mount /srv/ray-gcp
sudo -u raybench touch /srv/ray-gcp/write-check-{name}
""")


run_parallel(mount_worker, WORKERS)


## 6. Prepare the training runner
`GCPStudy` runs the benchmark and checks recovered model/optimizer/input state. All infrastructure setup above is ordinary notebook code.

In [ ]:
from gcp.launcher import GCPStudy
from gcp.suite import config

cloud = GCPStudy(STATE_FILE, WORKDIR / "selective-vm")
cloud.prepare(install_missing_dependencies=INSTALL_MISSING_DEPENDENCIES)


## 7. Smoke test: six observations
Run this before the comprehensive test. Failed gates need investigation; they do not mean replication is ineffective.

In [ ]:
SMOKE = "smoke-01"  # Use a new suite name to repeat with changed settings.
smoke_config = config(epochs=SMOKE_EPOCHS, checkpoint_every=SMOKE_CHECKPOINT_EVERY, profile_repeats=1)
smoke_config["keep_all_checkpoints"] = False
cloud.prepare_suite(SMOKE, smoke_config)
smoke_jobs = cloud.preview(SMOKE, smoke=True)


In [ ]:
smoke_summary = cloud.run_suite(SMOKE, smoke_jobs)
print(json.dumps(smoke_summary, indent=2))
assert smoke_summary["valid"], "Inspect the smoke results before continuing."


## 8. Comprehensive test
Five policies, varied prices/risks/deadlines and failure times, plus held-out failures. Review the observation count before running the next cell. Costs are normalized assumptions, not a GCP invoice.

In [ ]:
FULL = "full-01"
assert smoke_summary["valid"]
full_config = config(epochs=FULL_EPOCHS, checkpoint_every=FULL_CHECKPOINT_EVERY, profile_repeats=2)
full_config.update(
    keep_all_checkpoints=False,
    spot_cost_multipliers=PRICE_MULTIPLIERS,
    failure_probabilities=FAILURE_PROBABILITIES,
    deadline_multipliers=DEADLINE_MULTIPLIERS,
    fault_time_fractions=FAULT_TIME_FRACTIONS,
)
cloud.prepare_suite(FULL, full_config)
full_jobs = cloud.preview(FULL, repeats=REPEATS, case_indices=CASES)
assert len(full_jobs) <= MAX_OBSERVATIONS


In [ ]:
full_summary = cloud.run_suite(FULL, full_jobs, smoke_gate=SMOKE)
print(json.dumps(full_summary, indent=2))


## 9. Results
Small JSON results are saved locally after each observation. The optional download also copies remote checkpoint evidence before cleanup.

In [ ]:
print("Results:", cloud.local)
print("Send each suite's profile.json, search.json, replay.json, summary.json, jobs.json and manifest.json")
DOWNLOAD_CHECKPOINTS = False
if DOWNLOAD_CHECKPOINTS:
    for suite in [SMOKE, FULL]:
        print(cloud.download(suite))


## 10. Optional cleanup
Deletes these five VMs and their boot disks, including checkpoints on the head. Download wanted results first. The saved image is retained.

In [ ]:
DELETE_INSTANCES = False
if DELETE_INSTANCES:
    # Exact names and ownership label: never delete unrelated experiment VMs.
    existing = json.loads(gcloud("compute", "instances", "list", "--format=json"))
    targets = [vm for vm in existing if vm["name"] in NAMES and vm["zone"].rsplit("/", 1)[-1] == ZONE]
    for vm in targets:
        assert vm.get("labels", {}).get("ray-experiment") == RUN_ID, vm["name"]
    if targets:
        gcloud("compute", "instances", "delete", *[vm["name"] for vm in targets],
               f"--zone={ZONE}", "--delete-disks=boot", timeout=900)
    # Remove this run's two firewall rules. Keep the shared network/subnet.
    rules = json.loads(gcloud("compute", "firewall-rules", "list", "--format=json"))
    for rule in rules:
        if rule["name"] in [RUN_ID + "-internal", RUN_ID + "-iap"]:
            assert rule["network"].rsplit("/", 1)[-1] == NETWORK
            gcloud("compute", "firewall-rules", "delete", rule["name"])

else:
    print("Cleanup skipped. VMs/disks remain billable.")
